In [1]:
"""
need to pip install a few modules:
    pandas
    scikit-learn
    matplotlib
    xgboost
    catboost

"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
import model_helper_functions
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
#load data
data_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
# print(data_df.columns)
# print(test_df.columns)

Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
       'spectral_type', 'galaxy_population', 'class'],
      dtype='object')
Index(['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
       'spectral_type', 'galaxy_population'],
      dtype='object')


In [10]:
# total_len = len(data_df)  # three possible classifications: GALAXY, STAR, QSO
# data_df.head(10)
# test_df.head(10)
#pre-processing
data_df = model_helper_functions.feature_engineer(data_df)
    #per eda, drop rows where u < 10
data_df.drop(data_df[data_df['u'] < 10].index, inplace=True)
data_df.drop(columns=['id'], inplace=True)

test_df = model_helper_functions.feature_engineer(test_df)
    #per eda, drop rows where u < 10
test_df.drop(test_df[test_df['u'] < 10].index, inplace=True)
# test_df.drop(columns=['id'], inplace=True)


In [53]:
data_df.columns
# tmp = data_df.drop(columns=['id'], inplace=True)
# tmp.columns

Index(['u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population', 'class', 'X_loc', 'Y_loc', 'Z_loc', 'u_g_diff',
       'g_r_diff', 'r_i_diff', 'i_z_diff'],
      dtype='object')

In [11]:
#convert categorical variables to numeric dummy variables and do a test-train split
# import importlib
# importlib.reload(model_helper_functions)
if(True):
    cleaned_df = model_helper_functions.cat_to_dummy(data_df,False)
    X_train, X_test, y_train, y_test = train_test_split(cleaned_df.drop(columns=['class']), cleaned_df['class'], test_size=0.25, random_state = 12345)
else: #no need to convert categorical to dummy encoding
    X_train, X_test, y_train, y_test = train_test_split(data_df.drop(columns=['class']), data_df['class'], test_size=0.25, random_state = 12345)

In [15]:
print("Length of training data split: ", len(X_train), "and", len(y_train))
print("Length of testing section: ", len(X_test), "and", len(y_test))

Length of training data split:  433008 and 433008
Length of testing section:  144336 and 144336


In [13]:
###
# XGboost code, copied from https://xgboost.readthedocs.io/en/stable/get_started.html
# ToDo: hyperparameter tuning, read up on what is actually happening behind the scenes
###
depth_range = range(2, 7, 1)
n_estimators_range = range(1, 10, 1)
learning_rate_range = [0.01, 0.1, 0.5, 1]
for depth in depth_range:
    for n_estimators in n_estimators_range:
        for learning_rate in learning_rate_range:
            bst = XGBClassifier(n_estimators=n_estimators, max_depth=depth, learning_rate=learning_rate,enable_categorical=True)
            # fit model
            bst.fit(X_train, y_train)
            model_helper_functions.save_model_results(bst, X_test, y_test,"xgboost_validation"+str(depth)+"_"+str(n_estimators)+"_"+str(learning_rate),model_description="valid")
            model_helper_functions.save_model_results(bst, X_train, y_train,"xgboost_fit"+str(depth)+"_"+str(n_estimators)+"_"+str(learning_rate),model_description="fit")
            print("Completed depth: ", depth, "n_estimators: ", n_estimators, "learning_rate: ", learning_rate)

###
# Ideally visualize and debug more if time
###

Completed depth:  2 n_estimators:  1 learning_rate:  0.01
Completed depth:  2 n_estimators:  1 learning_rate:  0.1
Completed depth:  2 n_estimators:  1 learning_rate:  0.5
Completed depth:  2 n_estimators:  1 learning_rate:  1
Completed depth:  2 n_estimators:  2 learning_rate:  0.01
Completed depth:  2 n_estimators:  2 learning_rate:  0.1
Completed depth:  2 n_estimators:  2 learning_rate:  0.5
Completed depth:  2 n_estimators:  2 learning_rate:  1
Completed depth:  2 n_estimators:  3 learning_rate:  0.01
Completed depth:  2 n_estimators:  3 learning_rate:  0.1
Completed depth:  2 n_estimators:  3 learning_rate:  0.5
Completed depth:  2 n_estimators:  3 learning_rate:  1
Completed depth:  2 n_estimators:  4 learning_rate:  0.01
Completed depth:  2 n_estimators:  4 learning_rate:  0.1
Completed depth:  2 n_estimators:  4 learning_rate:  0.5
Completed depth:  2 n_estimators:  4 learning_rate:  1
Completed depth:  2 n_estimators:  5 learning_rate:  0.01
Completed depth:  2 n_estimators: 

0.9075282673761224

In [14]:
###
# catreg code
###
for depth in depth_range:
    for learning_rate in learning_rate_range:
        model = CatBoostClassifier(
            iterations=100,  #based on small experiments seems to be sufficient    
            learning_rate=learning_rate,   
            depth=depth,              
            verbose=0,
            cat_features=X_train.select_dtypes(include='category').columns.tolist()             
        )
        model.fit(X_train, y_train)
        model_helper_functions.save_model_results(model, X_test, y_test,"catboost_validation"+str(depth)+"_"+str(learning_rate),model_description="valid")
        model_helper_functions.save_model_results(model, X_train, y_train,"catboost_fit"+str(depth)+"_"+str(learning_rate),model_description="fit")
        print("Completed depth: ", depth, "learning_rate: ", learning_rate)


Completed depth:  2 learning_rate:  0.01
Completed depth:  2 learning_rate:  0.1
Completed depth:  2 learning_rate:  0.5
Completed depth:  2 learning_rate:  1
Completed depth:  3 learning_rate:  0.01
Completed depth:  3 learning_rate:  0.1
Completed depth:  3 learning_rate:  0.5
Completed depth:  3 learning_rate:  1
Completed depth:  4 learning_rate:  0.01
Completed depth:  4 learning_rate:  0.1
Completed depth:  4 learning_rate:  0.5
Completed depth:  4 learning_rate:  1
Completed depth:  5 learning_rate:  0.01
Completed depth:  5 learning_rate:  0.1
Completed depth:  5 learning_rate:  0.5
Completed depth:  5 learning_rate:  1
Completed depth:  6 learning_rate:  0.01
Completed depth:  6 learning_rate:  0.1
Completed depth:  6 learning_rate:  0.5
Completed depth:  6 learning_rate:  1


In [5]:
###
# Logistic regression code
###

first_model = LogisticRegression(class_weight='balanced', solver='lbfgs', max_iter=500)
first_model.fit(X_train, y_train)
# list out the model weights and intercepts for each class category:

# weights = first_model.coef_
# intercepts = first_model.intercept_

# # put all the values into a dataframe with relevant column names
# df_weights = pd.DataFrame(weights, columns=X_train.columns)
# df_weights["intercept"] = intercepts
# df_weights.index.name = "Class"

# print(df_weights)

model_helper_functions.save_model_results(first_model, X_test, y_test,"logreg_validation",model_description="valid")
model_helper_functions.save_model_results(first_model, X_train, y_train,"logreg_fit",model_description="fit")

c:\Users\rhals\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'balanced_accuracy': 0.9152234010446163,
 'per_class': {0: {'precision': 0.9798812245934959,
   'recall': 0.8725141568458336,
   'f1': 0.9230861284278785,
   'support': 282902},
  1: {'precision': 0.8696106032328472,
   'recall': 0.9356620983844618,
   'f1': 0.9014280015083368,
   'support': 88144},
  2: {'precision': 0.673378542862111,
   'recall': 0.9374939479035538,
   'f1': 0.783784330789937,
   'support': 61962}},
 'macro_precision': 0.8409567902294847,
 'macro_recall': 0.9152234010446163,
 'macro_f1': 0.8694328202420508,
 'weighted_precision': 0.9135747988123254,
 'weighted_recall': 0.8946670731256697,
 'weighted_f1': 0.8987437330808221,
 'matthews_corrcoef': 0.8150681415053233,
 'cohen_kappa': 0.8073730086289108,
 'roc_auc_ovr': 0.981338443300788,
 'confusion_matrix': array([[246836,  12027,  24039],
        [  1534,  82473,   4137],
        [  3534,    339,  58089]]),
 'y_pred': array([0, 1, 0, ..., 0, 0, 2], shape=(433008,))}

In [24]:
X_train.head()

,id,u,g,r,i,z,redshift,X_loc,Y_loc,Z_loc,u_g_diff,g_r_diff,r_i_diff,i_z_diff,A/F,G/K,M,O/B,Blue_Cloud,Red_Sequence
30829,30829,23.294060,22.499038,20.768159,19.878108,18.973470,0.526895,-0.848008,-0.198288,0.491492,0.795022,1.730879,0.890051,0.904638,0,0,1,0,0,1
485425,485425,24.674945,22.494397,22.011555,21.768530,21.464310,1.400081,-0.736674,0.441332,0.512385,2.180548,0.482842,0.243025,0.304220,1,0,0,0,0,1
142256,142256,22.885355,21.513540,20.346849,19.150284,18.742721,0.348395,-0.395798,-0.238303,0.886880,1.371815,1.166690,1.196565,0.407563,0,0,1,0,0,1
319751,319751,23.037464,23.544064,22.335594,21.717831,21.726273,1.171606,-0.832922,0.372072,0.409638,-0.506601,1.208470,0.617763,-0.008442,0,0,1,0,1,0
329195,329195,24.213389,22.810770,21.180478,19.581575,19.666788,0.270901,-0.585508,-0.720640,0.371292,1.402619,1.630292,1.598904,-0.085213,0,0,1,0,0,1


In [12]:
# Extract weights from all classes in the model
weights = np.array([estimator.coef_[0] for estimator in first_model.estimators_])

# Extract biases (intercepts) for all classes
biases = np.array([estimator.intercept_[0] for estimator in first_model.estimators_])

for target_class, w, b in zip(first_model.classes_, weights, biases):
    print(f"Class {target_class} -> Bias: {b:.4f} | Weights: {w}")

Class 0 -> Bias: 0.0002 | Weights: [ 7.99804608e-07  5.15928829e-03  4.05981495e-03  2.77064879e-03
  2.03760895e-03  1.67533799e-03 -5.52618560e-04 -2.02455326e-04
 -3.37116544e-05  1.82228778e-05  1.09947334e-03  1.28916617e-03
  7.33039840e-04  3.62270954e-04 -4.22256662e-04 -4.69087479e-05
  8.15210116e-04 -1.90909572e-04 -5.81371513e-04  7.36506648e-04]
Class 1 -> Bias: -0.0016 | Weights: [-1.24223596e-06 -5.41395810e-02 -2.39430662e-02  3.39303896e-03
  1.80355142e-02  2.39808444e-02  3.76031137e-02  3.71585198e-03
  5.41419545e-04  1.43866645e-03 -3.01965148e-02 -2.73361052e-02
 -1.46424752e-02 -5.94533015e-03  9.78441252e-03 -9.32909472e-04
 -1.65062522e-02  6.05644611e-03  1.46597686e-02 -1.62580717e-02]
Class 2 -> Bias: -0.0001 | Weights: [ 1.05826121e-07 -2.82475340e-02 -2.66732582e-02 -1.61674882e-02
 -9.65308762e-03 -5.45089791e-03 -1.27299195e-02  2.64239694e-03
  6.32055577e-04 -6.81937489e-04 -1.57427582e-03 -1.05057700e-02
 -6.51440054e-03 -4.20218971e-03  4.56984521e-

In [5]:
###
# iso fit best model
###

model = CatBoostClassifier(
        iterations=100,  #based on small experiments seems to be sufficient    
        learning_rate=0.5,   
        depth=6,              
        verbose=0,
        cat_features=X_train.select_dtypes(include='category').columns.tolist()             
    )
model.fit(X_train, y_train)

CatBoostClassifier(cat_features=['A/F', 'G/K', 'M', 'O/B', 'Blue_Cloud', 'Red_Sequence', 'spectral_type', 'galaxy_population'], depth=6, iterations=100, learning_rate=0.5, verbose=0)

In [13]:
#clean test data a lil
test_df = model_helper_functions.cat_to_dummy(test_df,False)

In [16]:
print(test_df.columns)
print(X_train.columns)

Index(['id', 'u', 'g', 'r', 'i', 'z', 'redshift', 'X_loc', 'Y_loc', 'Z_loc',
       'u_g_diff', 'g_r_diff', 'r_i_diff', 'i_z_diff', 'A/F', 'G/K', 'M',
       'O/B', 'Blue_Cloud', 'Red_Sequence', 'spectral_type',
       'galaxy_population'],
      dtype='object')
Index(['u', 'g', 'r', 'i', 'z', 'redshift', 'X_loc', 'Y_loc', 'Z_loc',
       'u_g_diff', 'g_r_diff', 'r_i_diff', 'i_z_diff', 'A/F', 'G/K', 'M',
       'O/B', 'Blue_Cloud', 'Red_Sequence', 'spectral_type',
       'galaxy_population'],
      dtype='object')


In [18]:
test_df_without_id = test_df.drop(columns=['id'])
final_predicts = model.predict(test_df_without_id)


In [47]:
# print(list(final_predicts)))
# print(final_predicts.shape)
# print(pd.Series(final_predicts[:,0]).head())
# print(len(test_df))
model_df = pd.DataFrame({'id': test_df['id'], 'class': pd.Series(final_predicts[:,0])})
model_df = model_helper_functions.dummy_to_cat(model_df)

model_df.to_csv("submissions/final_predictions.csv", index=False)

In [46]:
model_df.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [ ]:
###
# later wrap into a function, getting final result output for the test data set###


# set up dummy variables to ensure everything is numerical for the model
spectral = pd.get_dummies(test_df['spectral_type'], dtype=int)
galaxy_pop = pd.get_dummies(test_df['galaxy_population'], dtype=int)
    
# drop original categorical columns and concatenate the dummy variables to feed into logistic regression
temp_df = test_df.drop(columns=['spectral_type', 'galaxy_population'])
fin_test_df = pd.concat([temp_df, spectral, galaxy_pop], axis=1)

In [24]:
print(fin_test_df.head())

       id          u          g          r          i          z  redshift  \
0  577347  23.668066  21.951680  21.086183  20.180032  19.202124  0.429042   
1  577348  24.902933  22.338822  20.732163  19.860330  19.687691  0.867305   
2  577349  19.427591  18.474633  17.551314  16.570674  16.176765  0.224234   
3  577350  23.121029  21.526855  20.670159  20.417633  20.699095  0.066507   
4  577351  25.094282  22.643981  21.123173  19.439500  19.094158  0.977218   

      X_loc     Y_loc     Z_loc  u_g_diff  g_r_diff  r_i_diff  i_z_diff  A/F  \
0 -0.466950  0.785815  0.405528  1.716387  0.865497  0.906151  0.977908    0   
1 -0.572583 -0.470566  0.671354  2.564112  1.606658  0.871833  0.172640    0   
2 -0.993240  0.111959 -0.030650  0.952958  0.923319  0.980640  0.393909    0   
3 -0.996037 -0.085460 -0.024625  1.594174  0.856696  0.252526 -0.281462    0   
4 -0.711008 -0.651240  0.265243  2.450301  1.520808  1.683673  0.345342    0   

   G/K  M  O/B  Blue_Cloud  Red_Sequence  
0    1 

In [ ]:
# predict the classes of the testing data
final_predicts = first_model.predict(fin_test_df)


In [ ]:
# put the results into a DataFrame with 'id', can save to a csv and submit if needed

logis_df = pd.DataFrame({'id': fin_test_df['id'], 'class': final_predicts})
logis_df = model_helper_functions.dummy_to_cat(logis_df)

In [ ]:
# verify the resulting df looks correct
print(logis_df.head(20))

        id   class
0   577347  GALAXY
1   577348  GALAXY
2   577349  GALAXY
3   577350    STAR
4   577351  GALAXY
5   577352  GALAXY
6   577353  GALAXY
7   577354    STAR
8   577355  GALAXY
9   577356  GALAXY
10  577357    STAR
11  577358  GALAXY
12  577359     QSO
13  577360  GALAXY
14  577361     QSO
15  577362     QSO
16  577363    STAR
17  577364    STAR
18  577365     QSO
19  577366  GALAXY
